# Module 9: RAG & LLM Applications
## 06: Structured Output Extraction & Pydantic Validation

### Theoretical Overview
Unstructured natural language outputs break downstream microservices. Production LLM pipelines require strictly typed JSON objects. Strategies include:
1. **JSON Schema Constraints**: Enforcing schema validation at decoding time.
2. **Pydantic Validation**: Parsing responses into type-annotated dataclasses with runtime validators.
3. **Self-Correction & Fallback**: Automatically catching parse failures and prompting the model to repair invalid JSON.

In [ ]:
from pydantic import BaseModel, Field, EmailStr, ValidationError
from typing import List

class CandidateProfile(BaseModel):
    name: str = Field(description="Full legal name of candidate")
    email: EmailStr = Field(description="Contact email address")
    years_experience: float = Field(ge=0, description="Total verified years of professional experience")
    skills: List[str] = Field(default_factory=list, description="Primary technical competencies")
    hiring_recommendation: bool = Field(description="Whether candidate passes preliminary screening")

# Valid JSON payload simulation
raw_llm_json = """
{
  "name": "Sarah Connor",
  "email": "sarah.connor@cyberdyne.org",
  "years_experience": 8.5,
  "skills": ["Python", "PyTorch", "Kubernetes", "Vector Databases"],
  "hiring_recommendation": true
}
"""

profile = CandidateProfile.model_validate_json(raw_llm_json)
print(f"Validated Candidate: {profile.name}")
print(f"Skills: {', '.join(profile.skills)}")
print(f"Recommendation: {'HIRE' if profile.hiring_recommendation else 'PASS'}")

### Handling Malformed JSON and Repair Loops

In [ ]:
def parse_with_repair_simulation(raw_output: str):
    try:
        data = json.loads(raw_output)
        return data, "Clean parse"
    except json.JSONDecodeError as err:
        # Strip backticks and json tag
        cleaned = raw_output.strip().replace("```json", "").replace("```", "").strip()
        try:
            data = json.loads(cleaned)
            return data, "Repaired markdown wrapper"
        except Exception as e:
            return None, f"Failed repair: {e}"

test_raw = "```json\n{\"action\": \"search\", \"query\": \"RAG architecture\"}\n```"
parsed, msg = parse_with_repair_simulation(test_raw)
print("Parse status:", msg)
print("Output:", parsed)